# EloSense v4: Chess Form Volatility

EloSense so far has been about predicting a player's rating band from a single game or a stack of aggregated games. This version asks a different question: not what a player's skill level is, but how erratic their performance currently is, separate from skill.

The idea comes straight out of quant finance. A stock's price level and its volatility are two different things you track separately, and volatility itself is treated as a hidden state that drifts over time, usually estimated with a stochastic volatility (SV) model fit by particle filtering. I'm applying that same machinery to chess: define a per-game "surprise" signal (did the player do better or worse than expected), treat the surprise's volatility as a hidden state `h_t`, and estimate `h_t` with a bootstrap particle filter plus Particle Marginal Metropolis-Hastings (PMMH) for the model parameters.

This reuses the same 60k-game Chess.com dataset and the player-disjoint split built in `elosense_v3.ipynb`, instead of starting over. Like v3, this notebook doesn't modify v1-v3, it re-derives whatever it needs from the cached features.

Sections below follow the W1-W5 work breakdown I used to plan this: data audit, observation signal, particle filter + PMMH (including the hierarchical extension), feature ablation against the existing v1-v3 model, and a correlation/trend pass at the end.

## W1. Sequencing and data audit

### 1. Chronological game ordering per player

The dataset has one row per game with a `white_id` and `black_id`, not one row per player-game in time order. To build a volatility path for a player I need their games sorted by when they were actually played, from either side of the board.

The PGN text embeds `UTCDate` and `UTCTime` tags, so I pull those out with a regex instead of relying on row order in the CSV (row order turns out to already be roughly chronological per player, but I don't want to depend on that).

In [1]:
import pandas as pd
import numpy as np
import re

DATE_RE = re.compile(r'\[UTCDate "([\d.]+)"\]')
TIME_RE = re.compile(r'\[UTCTime "([\d:]+)"\]')

RESULT_TO_SCORE = {
    "win": 1.0, "checkmated": 0.0, "agreed": 0.5, "repetition": 0.5,
    "stalemate": 0.5, "insufficient": 0.5, "50move": 0.5, "timevsinsufficient": 0.5,
    "abandoned": 0.0, "resigned": 0.0, "timeout": 0.0, "kingofthehill": 0.0,
    "threecheck": 0.0,
}

df = pd.read_csv("../data/club_games_data.csv")
print(f"rows: {len(df)}")

def player_history(pid, df=df):
    """Full chronological game log for one player, either side of the board."""
    mask = (df["white_id"] == pid) | (df["black_id"] == pid)
    sub = df.loc[mask].copy()
    is_white = (sub["white_id"] == pid).values
    sub["player_rating"] = np.where(is_white, sub["white_rating"], sub["black_rating"])
    sub["opp_rating"] = np.where(is_white, sub["black_rating"], sub["white_rating"])
    result = np.where(is_white, sub["white_result"], sub["black_result"])
    sub["actual_score"] = pd.Series(result, index=sub.index).map(RESULT_TO_SCORE)

    dates = sub["pgn"].str.extract(DATE_RE)[0]
    times = sub["pgn"].str.extract(TIME_RE)[0]
    sub["dt"] = pd.to_datetime(dates + " " + times, format="%Y.%m.%d %H:%M:%S")
    sub = sub.sort_values("dt").reset_index(drop=True)

    sub["expected_score"] = 1.0 / (1.0 + 10 ** ((sub["opp_rating"] - sub["player_rating"]) / 400.0))
    sub["x"] = sub["actual_score"] - sub["expected_score"]
    return sub[["dt", "player_rating", "opp_rating", "actual_score", "expected_score", "x", "time_class", "rated"]]

# sanity check on the most active player in the dataset
sample_pid = "https://api.chess.com/pub/player/aalisyed"
h = player_history(sample_pid)
print(f"{sample_pid}: {len(h)} games, {h['dt'].min()} to {h['dt'].max()}")
assert h["dt"].is_monotonic_increasing, "sort failed"
assert h["x"].isna().sum() == 0, "unmapped result code found"
h.head()

rows: 66879
https://api.chess.com/pub/player/aalisyed: 2253 games, 2021-05-03 21:10:10 to 2021-05-26 03:17:10


,dt,player_rating,opp_rating,actual_score,expected_score,x,time_class,rated
0,2021-05-03 21:10:10,1733,1566,1.0,0.723388,0.276612,bullet,True
1,2021-05-03 21:13:14,1738,1588,1.0,0.703385,0.296615,bullet,True
2,2021-05-03 21:15:21,1732,1819,0.0,0.377350,-0.377350,bullet,True
3,2021-05-03 21:17:47,1736,1556,1.0,0.738109,0.261891,bullet,True
4,2021-05-03 21:20:33,1725,1619,0.0,0.647983,-0.647983,bullet,True
